<a href="https://colab.research.google.com/github/lricci03/challengedata/blob/main/CFM2024-new/create_submission.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Create submission

This is a template to create the submission file for the challenge.

The submission is a csv file containing the prediction for the test set.

Workflow:

Once we have finished model selection:
- Upload the full train set and preprocess it;
- retrain the chosen architecture on the full training set;
- Upload the test set, and preprocessed;
- predict on test.

In [1]:
import torch
import torch.nn as nn

if torch.cuda.is_available():
  device = 'cuda'
elif torch.backends.mps.is_available():
  device = 'mps'
else:
  device = 'cpu'

In [2]:
import numpy as np

In [3]:
from google.colab import drive

# drive.flush_and_unmount()
drive.mount("/content/drive")

Mounted at /content/drive


## Upload the full train set

### Train set sequential features

In [ ]:
path = Path("/content/drive/MyDrive/Challengedata/Xtrain.parquet")

df_features = pd.read_parquet(path)

df_features.head()

In [ ]:
df_targets = pd.read_csv("/content/drive/MyDrive/Challengedata/y_train_or6m3Ta.csv")

df_targets.head()

In [ ]:
def add_order_features(df):

    # how many previous events involved this same order
    df["order_previous_count"] = (
        df.groupby(["obs_id", "order_id"])
          .cumcount()
          .astype("float32")
    )

    # whether this order was seen before
    df["order_seen_before"] = (
        df["order_previous_count"] > 0
    ).astype("float32")

    # raw ID no longer needed
    df.drop("order_id", axis=1, inplace=True)

    return df

In [ ]:
df_features = add_order_features(df_features)

In [ ]:
df_features = df_features[[
    "obs_id",
    "price", "bid", "ask",
    "bid_size", "ask_size",
    "flux",
    "action", "venue", "side", "trade",
    "order_previous_count",
    "order_seen_before"
]]

In [ ]:
num_attribs = ['price', 'bid', 'ask', 'bid_size', 'ask_size', 'flux', 'trade', 'order_seen_before', 'order_previous_count']
cat_attribs = ['venue', 'action', 'side']

In [ ]:
df_features[num_attribs] = df_features[num_attribs].astype("float32")

In [ ]:
df_features = pd.get_dummies(df_features, columns=cat_attribs, dtype="float32")

In [ ]:
df_features = df_features.set_index("obs_id")
df_targets = df_targets.set_index("obs_id")

In [ ]:
n_features = X_train.shape[1]

print(n_features)

Convert to tensors

In [ ]:
X_seq = torch.tensor(
    df_features.to_numpy(dtype=np.float32),
    dtype=torch.float32
).reshape(-1, 100, n_features)

y_train_full = torch.tensor(
    targets["eqt_code_cat"].to_numpy(),
    dtype=torch.long
)

### Train set engineered features

In [ ]:
df_eng = pd.read_csv("/content/drive/MyDrive/Challengedata/eng_features.csv")

In [ ]:
df_eng = df_eng.set_index("obs_id")

In [ ]:
X_eng = torch.tensor(df_eng.values, dtype=torch.float32)

In [ ]:
mean = X_eng.mean(dim=0, keepdim=True)
std = X_eng.std(dim=0, keepdim=True)

# avoid division by zero for constant features
std[std == 0] = 1.0

X_eng_std = (X_eng - mean) / std

## Datasets and DataLoaders

Dataset to combine sequential and engineered features

In [ ]:
from torch.utils.data import Dataset
import torch

class StockDataset(Dataset):
    def __init__(self, X_seq, X_eng, y):
        self.X_seq = X_seq
        self.X_eng = X_eng
        self.y = y

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return (
            self.X_seq[idx],
            self.X_eng[idx],
            self.y[idx]
        )

In [ ]:
train_dataset = StockDataset(X_seq, X_eng, y_train_full)

In [ ]:
from torch.utils.data import DataLoader

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

**Sequential features only**

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

train_dataset_seq = TensorDataset(X_seq, y_train_full)
train_loader_seq = DataLoader(train_dataset_seq, batch_size=64, shuffle=True)

## Train function

We don't need the validation and the best model logic we used during model selection.

We train on the number of epochs we selected during model selection.

**Train function for sequential data only**

In [ ]:
%pip install -q torchmetrics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 8.2 MB/s eta 0:00:00


In [ ]:
import torchmetrics


def train_full(model, optimizer, loss_fn, metric, train_loader, n_epochs):

    history = {
        "train_losses": [],
        "train_metrics": []
    }

    for epoch in range(n_epochs):
        total_loss = 0.0
        metric.reset()
        model.train()

        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)

            y_pred = model(X_batch)
            loss = loss_fn(y_pred, y_batch)

            total_loss += loss.item()

            loss.backward()
            optimizer.step()
            optimizer.zero_grad()

            y_class = y_pred.detach().argmax(dim=1)
            metric.update(y_class, y_batch)

        train_loss = total_loss / len(train_loader)
        train_metric = metric.compute().item()

        history["train_losses"].append(train_loss)
        history["train_metrics"].append(train_metric)

        print(
            f"Epoch {epoch + 1}/{n_epochs}, "
            f"train loss: {train_loss:.4f}, "
            f"train metric: {train_metric:.4f}"
        )

    return history

**Train function for joint model**

In [ ]:
def train_full_joint(model, optimizer, loss_fn, metric, train_loader, n_epochs):

    history = {
        "train_losses": [],
        "train_metrics": []
    }

    for epoch in range(n_epochs):
        total_loss = 0.0
        metric.reset()
        model.train()

        for X_seq_batch, X_eng_batch, y_batch in train_loader:
            X_seq_batch = X_seq_batch.to(device)
            X_eng_batch = X_eng_batch.to(device)
            y_batch = y_batch.to(device)

            optimizer.zero_grad()

            y_pred = model(X_seq_batch, X_eng_batch)
            loss = loss_fn(y_pred, y_batch)

            total_loss += loss.item()

            loss.backward()
            optimizer.step()

            y_class = y_pred.detach().argmax(dim=1)
            metric.update(y_class, y_batch)

        train_loss = total_loss / len(train_loader)
        train_metric = metric.compute().item()

        history["train_losses"].append(train_loss)
        history["train_metrics"].append(train_metric)

        print(
            f"Epoch {epoch + 1}/{n_epochs}, "
            f"train loss: {train_loss:.4f}, "
            f"train metric: {train_metric:.4f}"
        )

    return history

## Train on the full training set

The best model is a Deep GRU (2 gru layers) that we trained on the sequential features only (so not joint), and the features are not normalized.

It is saved as `deep_gru_model_128.pt`

In [ ]:
class DeepGRUModel(nn.Module):
  def __init__(self, input_size, hidden_size, output_size):
    super().__init__()
    self.hidden_size = hidden_size
    self.rnn = nn.GRU(input_size, hidden_size, num_layers=2, batch_first=True)
    self.output = nn.Linear(hidden_size, output_size)

  def forward(self,X):
    outputs, last_state = self.rnn(X)
    return self.output(outputs[:, -1])

In [ ]:
model = DeepGRUModel(input_size=n_features, hidden_size=128, output_size=24).to(device)


model.load_state_dict(
    torch.load("/content/drive/MyDrive/Challengedata/deep_gru_model_128.pt")
)

RuntimeError: Attempting to deserialize object on a CUDA device but torch.cuda.is_available() is False. If you are running on a CPU-only machine, please use torch.load with map_location=torch.device('cpu') to map your storages to the CPU.

## Upload the test set

## Predict

In [ ]:
model.eval()

predictions = []

with torch.no_grad():
    for (X_batch,) in test_loader:
        X_batch = X_batch.to(device)

        logits = model(X_batch)
        y_pred = logits.argmax(dim=1)

        predictions.extend(y_pred.cpu().numpy())

Save the predictions in csv

In [ ]:
submission = pd.DataFrame({
    "obs_id": range(len(predictions)),
    "eqt_code_cat": predictions
})

submission.to_csv(
    "/content/drive/MyDrive/Challengedata/submission.csv",
    index=False
)